# 04 — Baseline YOLO26n sur PitchLens v002

Ce notebook entraîne un premier détecteur sur tout le train de v002, le compare au modèle préentraîné sur toute la validation, puis produit les métriques, matrices, courbes de loss et exemples visuels.

Les images restent dans `data/processed/v002`. Chaque exécution crée uniquement un dossier daté contenant les poids et rapports dans `models/checkpoints/baseline_runs/`. Le split test n’est jamais utilisé ici.


## 1. Environnement et fichiers

Utilise le kernel Python du projet (`.venv`). Les dépendances sont déclarées dans l'extra `training` de pyproject.toml. La cellule affiche leurs versions et vérifie l'empreinte du modèle téléchargé.


In [ ]:
from pathlib import Path
import csv
from datetime import datetime
import hashlib
import json
import os
import random
import shutil
import tempfile
import time
from importlib.metadata import version

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.environ["YOLO_CONFIG_DIR"] = str(PROJECT_ROOT / ".ultralytics")
os.environ["YOLO_AUTOINSTALL"] = "false"
os.environ["MPLBACKEND"] = "Agg"

import torch
from ultralytics import YOLO, settings
from IPython.display import Image as DisplayImage, display

settings.update({"sync": False, "wandb": False, "mlflow": False, "clearml": False})
DATA_ROOT = PROJECT_ROOT / "data/processed/v002"
WEIGHTS = PROJECT_ROOT / "models/checkpoints/yolo26n.pt"
provenance = json.loads((PROJECT_ROOT / "models/reports/yolo26n_pretrained.json").read_text())
assert hashlib.sha256(WEIGHTS.read_bytes()).hexdigest() == provenance["sha256"]
dataset_report = json.loads((DATA_ROOT / "build_report.json").read_text())
assert dataset_report["images"] == 4549
versions = {name: version(name) for name in ("ultralytics", "torch", "torchvision")}
print(versions)
print("MPS disponible :", torch.backends.mps.is_available())
print("Dataset :", DATA_ROOT)


## 2. Entraînement complet et comparaison avant/après

Nous utilisons **tout le train de v002 (3 048 images)** et **toute la validation (670 images)**. On repart toujours des poids préentraînés `yolo26n.pt`.

Objectif local : au plus 5 époques avec une limite indicative de 10 minutes. Ultralytics vérifie cette limite entre certaines étapes et peut donc terminer l’époque en cours. Le GPU Apple est utilisé si disponible. Un véritable entraînement plus long sera exécuté ensuite sur GPU Hugging Face.

Les résultats portent uniquement sur la validation. Le test reste réservé à l’évaluation finale du modèle choisi.


In [ ]:
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from pitchlens.training.evaluate import BallValidator
from pitchlens.training.diagnostics import change_category, detection_counts, quiet_ultralytics_progress

torch.set_num_threads(4)
full_parent = PROJECT_ROOT / "models/checkpoints/baseline_runs"
full_parent.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
FULL_ROOT = full_parent / f"yolo26n_v002_local_{run_timestamp}"
FULL_ROOT.mkdir()
full_data = DATA_ROOT  # référence v002 directement : aucune copie des images
full_yaml = FULL_ROOT / "dataset.yaml"
full_yaml.write_text(
    "path: " + json.dumps(str(DATA_ROOT)) +
    "\ntrain: train/images\nval: val/images\nnames:\n  0: ball\n"
)
TRAIN_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
EVAL_DEVICE = "cpu"  # même matériel pour les deux évaluations
evaluation_config = dict(
    data=str(full_yaml), split="val", imgsz=640, batch=4,
    device=EVAL_DEVICE, workers=0, conf=0.001, iou=0.7, max_det=300,
    plots=True, verbose=False, project=str(FULL_ROOT),
)
print("Train :", TRAIN_DEVICE, "— comparaison :", EVAL_DEVICE)
print("Expérience :", FULL_ROOT)


### 2.1 Avant : que sait faire le modèle préentraîné ?

YOLO26n possède une classe COCO `sports ball`. Notre évaluateur conserve uniquement ces prédictions et les compare à la classe `ball=0` de v002, images négatives incluses.

La mAP utilise tous les scores à partir de 0,001. Les chiffres opérationnels utilisent **confiance ≥ 0,25 et IoU ≥ 0,50**, identiques avant et après.


In [ ]:
before_model = YOLO(str(WEIGHTS))
with quiet_ultralytics_progress():
    before_metrics = before_model.val(
        validator=BallValidator, name="before", **evaluation_config
    )
def evaluation_summary(result):
    return {
        "map50": float(result.box.map50),
        "map50_95": float(result.box.map),
        "fixed": result.fixed_threshold,
        "speed_ms_per_image": dict(result.speed),
    }
before_summary = evaluation_summary(before_metrics)
(FULL_ROOT / "before.json").write_text(json.dumps(before_summary, indent=2))
display(before_summary)


### 2.2 Entraîner sur v002

Les hyperparamètres restent simples : AdamW, résolution 640, graine 42 et validation à chaque époque. Les augmentations standards restent actives ; mosaic et mixup sont désactivés pour ce baseline.

Les détails vont dans `training.log`. Le notebook n’affiche qu’un résumé par époque. Les images et labels ne sont pas copiés dans le run.


In [ ]:
import logging
import gc
from ultralytics.utils import LOGGER

del before_model
gc.collect()
full_model = YOLO(str(WEIGHTS))
full_config = dict(
    data=str(full_yaml), epochs=12, time=0.75, imgsz=640, patience = 4,
    batch=4, device=TRAIN_DEVICE, workers=0, seed=42,
    deterministic=True, optimizer="AdamW", lr0=0.001,
    amp=False, cache=False, mosaic=0.0, mixup=0.0, close_mosaic=0,
    val=True, plots=True, save=True, verbose=False,
    project=str(FULL_ROOT), name="train", exist_ok=False,
)
(FULL_ROOT / "experiment.json").write_text(json.dumps({
    "training": full_config, "evaluation": evaluation_config,
    "versions": versions, "pretrained_sha256": provenance["sha256"],
    "dataset_decisions_sha256": dataset_report["decisions_sha256"],
    "comparison_confidence": 0.25, "comparison_iou": 0.5,
    "test_split_used": False,
}, indent=2))
def report_epoch(trainer):
    print(f"Époque {trainer.epoch + 1} terminée — "
          f"mAP50 validation : {trainer.metrics.get('metrics/mAP50(B)', 0):.1%}")
full_model.add_callback("on_fit_epoch_end", report_epoch)
old_handlers = list(LOGGER.handlers)
log_handler = logging.FileHandler(FULL_ROOT / "training.log")
LOGGER.handlers = [log_handler]
full_started = time.monotonic()
try:
    with quiet_ultralytics_progress():
        full_model.train(**full_config)
finally:
    LOGGER.handlers = old_handlers
    log_handler.close()
FULL_SAVE_DIR = Path(full_model.trainer.save_dir)
full_active_seconds = time.monotonic() - full_started
print(f"Temps actif observé : {full_active_seconds/60:.1f} minutes — {FULL_SAVE_DIR}")


In [ ]:
full_best = FULL_SAVE_DIR / "weights/best.pt"
assert full_best.is_file()

# supprimer full_model seulement s'il existe (évite NameError si la cellule d'entraînement
# n'a pas été exécutée ou si la variable a déjà été supprimée)
if "full_model" in globals():
    try:
        del full_model
    except Exception:
        pass

gc.collect()
if TRAIN_DEVICE == "mps":
    try:
        torch.mps.empty_cache()
    except Exception:
        pass

after_model = YOLO(str(full_best))
with quiet_ultralytics_progress():
    after_metrics = after_model.val(
        validator=BallValidator, name="after", **evaluation_config
    )
after_summary = evaluation_summary(after_metrics)

with (FULL_SAVE_DIR / "results.csv").open() as stream:
    history = [{k.strip(): float(v) for k, v in row.items()} for row in csv.DictReader(stream)]

comparison = {
    "before": before_summary, "after": after_summary,
    "epochs_completed": len(history),
    "training_seconds_ultralytics": history[-1]["time"],
    "active_seconds_observed": full_active_seconds,
    "best_weights": str(full_best), "validation_images": 670,
    "training_images": 3048, "test_split_used": False,
}
(FULL_ROOT / "comparison.json").write_text(json.dumps(comparison, indent=2))


## 3. Bilan lisible

- **Précision** : parmi les détections, quelle proportion est correcte ?
- **Rappel** : parmi les ballons annotés, quelle proportion est retrouvée ?
- **F1** : compromis entre précision et rappel.
- **mAP50 / mAP50–95** : qualité sur plusieurs seuils de confiance et de recouvrement.
- **FP / FN** : fausses détections et ballons ratés au seuil fixé.


In [ ]:
rows = [
    ("Précision (conf ≥ 0,25)", before_summary["fixed"]["precision"], after_summary["fixed"]["precision"]),
    ("Rappel (conf ≥ 0,25)", before_summary["fixed"]["recall"], after_summary["fixed"]["recall"]),
    ("F1", before_summary["fixed"]["f1"], after_summary["fixed"]["f1"]),
    ("mAP50", before_summary["map50"], after_summary["map50"]),
    ("mAP50–95", before_summary["map50_95"], after_summary["map50_95"]),
]
table = ["| Mesure | Avant | Après | Écart |", "|---|---:|---:|---:|"]
for label, before_value, after_value in rows:
    table.append(f"| {label} | {before_value:.1%} | {after_value:.1%} | {(after_value-before_value)*100:+.1f} pts |")
for label, key in (("Bonnes détections", "tp"), ("Fausses détections", "fp"), ("Ballons ratés", "fn"), ("Fausses détections sur négatifs", "fp_on_negatives")):
    a, b = before_summary["fixed"].get(key, 0), after_summary["fixed"].get(key, 0)
    table.append(f"| {label} | {a} | {b} | {b-a:+d} |")
display(Markdown("\n".join(table)))
(FULL_ROOT / "comparison.md").write_text("\n".join(table))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = list(range(len(rows)))
axes[0].bar([i-0.18 for i in x], [r[1]*100 for r in rows], width=.36, label="Avant")
axes[0].bar([i+0.18 for i in x], [r[2]*100 for r in rows], width=.36, label="Après")
axes[0].set_xticks(x, ["Précision", "Rappel", "F1", "mAP50", "mAP50–95"])
axes[0].set_ylim(0,100)
axes[0].set_ylabel("%")
axes[0].set_title("Même validation, même protocole")
axes[0].legend()
error_rows = [("TP", "tp"), ("FP", "fp"), ("FN", "fn"), ("FP négatifs", "fp_on_negatives")]
error_x = list(range(len(error_rows)))
axes[1].bar([i-0.18 for i in error_x], [before_summary["fixed"].get(k, 0) for _, k in error_rows], width=.36, label="Avant")
axes[1].bar([i+0.18 for i in error_x], [after_summary["fixed"].get(k, 0) for _, k in error_rows], width=.36, label="Après")
axes[1].set_xticks(error_x, [label for label, _ in error_rows])
axes[1].set_ylabel("Nombre de boîtes")
axes[1].set_title("Détections et erreurs au seuil fixé")
axes[1].legend()
fig.tight_layout()
comparison_path = FULL_ROOT / "comparison.png"
fig.savefig(comparison_path, dpi=150)
plt.close(fig)
display(DisplayImage(filename=str(comparison_path)))


## 8. Matrice de confusion et courbes d'apprentissage

Pour un détecteur à une seule classe, la matrice contient :

- **TP** : ballon correctement détecté ;
- **FP** : boîte prédite qui ne correspond à aucun ballon ;
- **FN** : ballon annoté mais raté ;
- la case **vrai fond / fond prédit** n'est pas définie, car une image contient une infinité de positions de fond. Il n'existe donc pas de nombre naturel de vrais négatifs au niveau des boîtes.

La matrice ci-dessous reprend exactement le même point de fonctionnement que le tableau : confiance ≥ 0,25 et IoU ≥ 0,50. Les matrices automatiques produites par Ultralytics servent à la mAP et utilisent un seuil beaucoup plus bas ; elles ne doivent pas être comparées directement à ce tableau.

Les courbes train/validation permettent ensuite de surveiller le surapprentissage : si la perte train continue de baisser tandis que la perte validation remonte durablement, le modèle mémorise probablement le train. Deux époques donnent seulement une tendance ; il en faut davantage pour conclure.

In [ ]:
import numpy as np

# Matrice de confusion de détection au seuil opérationnel fixé.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, title, summary in zip(axes, ("Avant", "Après"), (before_summary, after_summary)):
    fixed = summary["fixed"]
    matrix = np.array([[fixed["tp"], fixed["fp"]], [fixed["fn"], np.nan]])
    masked = np.ma.masked_invalid(matrix)
    axis.imshow(masked, cmap="Blues")
    for row in range(2):
        for column in range(2):
            value = matrix[row, column]
            axis.text(column, row, "N/A" if np.isnan(value) else str(int(value)),
                      ha="center", va="center", fontsize=14,
                      color="white" if not np.isnan(value) and value > np.nanmax(matrix) / 2 else "black")
    axis.set_xticks((0, 1), ("Vrai ballon", "Vrai fond"))
    axis.set_yticks((0, 1), ("Prédit ballon", "Prédit fond"))
    axis.set_title(title)
fig.suptitle("Matrice de confusion — confiance 0,25 / IoU 0,50")
fig.tight_layout()
confusion_path = FULL_ROOT / "confusion_matrix_fixed.png"
fig.savefig(confusion_path, dpi=150)
plt.close(fig)
display(DisplayImage(filename=str(confusion_path)))

# Courbes d'apprentissage : chaque perte train est comparée à son équivalent validation.
epochs = [row["epoch"] for row in history]
losses = (("box_loss", "Localisation"), ("cls_loss", "Classification"), ("l1_loss", "L1"))
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for axis, (key, title) in zip(axes.flat[:3], losses):
    axis.plot(epochs, [row[f"train/{key}"] for row in history], marker="o", label="Train")
    axis.plot(epochs, [row[f"val/{key}"] for row in history], marker="o", label="Validation")
    axis.set_title(f"Perte {title}")
    axis.set_xlabel("Époque")
    axis.set_ylabel("Loss")
    axis.legend()
axes[1, 1].plot(epochs, [row["metrics/mAP50(B)"] * 100 for row in history], marker="o", label="mAP50")
axes[1, 1].plot(epochs, [row["metrics/mAP50-95(B)"] * 100 for row in history], marker="o", label="mAP50–95")
axes[1, 1].set_title("Qualité sur validation")
axes[1, 1].set_xlabel("Époque")
axes[1, 1].set_ylabel("%")
axes[1, 1].legend()
fig.suptitle("Évolution de l'apprentissage")
fig.tight_layout()
learning_curves_path = FULL_ROOT / "learning_curves.png"
fig.savefig(learning_curves_path, dpi=150)
plt.close(fig)
display(DisplayImage(filename=str(learning_curves_path)))

if len(history) < 4:
    trend_note = ("Seulement " + str(len(history)) + " époques : les pertes train et validation baissent, "
                  "mais il est trop tôt pour conclure sur le surapprentissage.")
else:
    train_down = history[-1]["train/box_loss"] < history[-2]["train/box_loss"]
    val_up = history[-1]["val/box_loss"] > history[-2]["val/box_loss"]
    trend_note = ("Alerte possible : train baisse mais validation remonte." if train_down and val_up
                  else "Pas de divergence train/validation sur la dernière époque.")
display(Markdown("**Lecture :** " + trend_note))

## 5. Courbes de décision produites par YOLO

Le dossier du run contient plusieurs graphiques automatiques. Les quatre utiles pour choisir le seuil de confiance sont regroupés ici :

- **Precision–Recall** résume le compromis global et son aire correspond à la mAP50 ;
- **F1–Confidence** indique le seuil donnant le meilleur compromis précision/rappel ;
- **Precision–Confidence** et **Recall–Confidence** montrent ce que l’on gagne et perd lorsque le seuil augmente.

Les autres fichiers automatiques restent disponibles comme artefacts techniques. `results.png` est volontairement omis car nos courbes d’apprentissage sont plus lisibles ; les matrices Ultralytics utilisent un seuil différent, donc nous présentons plutôt notre matrice fixe cohérente avec le tableau.

In [ ]:
curve_files = [
    (FULL_ROOT / "after/BoxPR_curve.png", "Précision–rappel"),
    (FULL_ROOT / "after/BoxF1_curve.png", "F1 selon la confiance"),
    (FULL_ROOT / "after/BoxP_curve.png", "Précision selon la confiance"),
    (FULL_ROOT / "after/BoxR_curve.png", "Rappel selon la confiance"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, (curve_path, title) in zip(axes.flat, curve_files):
    axis.imshow(Image.open(curve_path))
    axis.set_title(title)
    axis.axis("off")
fig.suptitle("Choisir le seuil de confiance du modèle entraîné")
fig.tight_layout()
decision_curves_path = FULL_ROOT / "decision_curves.png"
fig.savefig(decision_curves_path, dpi=140)
plt.close(fig)
display(DisplayImage(filename=str(decision_curves_path)))

display(Markdown("### Ce que le modèle a réellement reçu pendant l’entraînement"))
display(Markdown("Ce graphique décrit les positions et dimensions des boîtes du train. Il sert surtout à repérer un dataset très déséquilibré."))
display(DisplayImage(filename=str(FULL_SAVE_DIR / "labels.jpg"), width=850))

## 6. Exemples visuels avant/après

La galerie utilise 32 images déterministes, uniquement pour l’inspection qualitative. Les métriques restent calculées sur les 670 images de validation.

Les résultats sont séparés en trois planches courtes :

1. **améliorations réussies** : une erreur du modèle préentraîné a disparu ;
2. **ballons encore ratés** : une annotation verte n’a aucune prédiction correspondante ;
3. **fausses alertes restantes** : une boîte rouge ne correspond à aucun ballon annoté.

**Vert = annotation de référence. Rouge = prédiction du modèle.** Une superposition correcte peut faire apparaître les deux couleurs très proches ; les titres décrivent donc aussi le résultat en français courant.

In [ ]:
from collections import Counter
from PIL import Image, ImageDraw
import gc

VISUAL_SAMPLE_SIZE = 32
VISUAL_CHUNK_SIZE = 4
VISUAL_BATCH_SIZE = 1

def ground_truth_boxes(image_path):
    with Image.open(image_path) as image:
        image_width, image_height = image.size
    boxes = []
    label_path = full_data / "val/labels" / (image_path.stem + ".txt")
    for line in label_path.read_text().splitlines():
        _, x, y, width, height = map(float, line.split())
        boxes.append(((x-width/2)*image_width, (y-height/2)*image_height,
                      (x+width/2)*image_width, (y+height/2)*image_height))
    return np.asarray(boxes, dtype=float).reshape(-1, 4)

def compact_predictions(model, image_paths, classes=None):
    """Infer small chunks and retain boxes/confidences, never decoded images."""
    compact = []
    with quiet_ultralytics_progress():
        for start in range(0, len(image_paths), VISUAL_CHUNK_SIZE):
            chunk = image_paths[start:start + VISUAL_CHUNK_SIZE]
            results = model.predict(
                [str(path) for path in chunk], classes=classes, conf=.25,
                imgsz=640, batch=VISUAL_BATCH_SIZE, device="cpu", verbose=False,
            )
            compact.extend({
                "boxes": result.boxes.xyxy.cpu().numpy().copy(),
                "confidence": result.boxes.conf.cpu().numpy().copy(),
            } for result in results)
            del results
            gc.collect()
    return compact

all_validation_paths = sorted((full_data / "val/images").glob("*.jpg"))
validation_paths = random.Random(42).sample(all_validation_paths, min(VISUAL_SAMPLE_SIZE, len(all_validation_paths)))
original = YOLO(str(WEIGHTS))
sports_class = next(key for key, value in original.names.items() if value == "sports ball")
before_predictions = compact_predictions(original, validation_paths, [sports_class])
del original
gc.collect()
after_predictions = compact_predictions(after_model, validation_paths)

records = []
for image_path, before_prediction, after_prediction in zip(validation_paths, before_predictions, after_predictions):
    truth = ground_truth_boxes(image_path)
    before_boxes = before_prediction["boxes"]
    after_boxes = after_prediction["boxes"]
    before_counts = detection_counts(truth, before_boxes)
    after_counts = detection_counts(truth, after_boxes)
    records.append({
        "path": image_path, "truth": truth,
        "before_boxes": before_boxes, "before_conf": before_prediction["confidence"],
        "after_boxes": after_boxes, "after_conf": after_prediction["confidence"],
        "before": before_counts, "after": after_counts,
        "category": change_category(before_counts, after_counts),
    })

del before_predictions, after_predictions
category_counts = Counter(record["category"] for record in records)
display(category_counts)
def describe_counts(counts):
    parts = []
    if counts["tp"]:
        parts.append(f"{counts['tp']} ballon(s) trouvé(s)")
    if counts["fn"]:
        parts.append(f"{counts['fn']} ballon(s) raté(s)")
    if counts["fp"]:
        parts.append(f"{counts['fp']} fausse(s) alerte(s)")
    return ", ".join(parts) if parts else "aucun ballon et aucune fausse alerte"

def choose_examples(predicate, limit=2, excluded=None):
    excluded = excluded or set()
    candidates = [record for record in records if predicate(record) and record["path"].name not in excluded]
    candidates.sort(key=lambda record: (record["after"]["fp"] + record["after"]["fn"], record["path"].name))
    chosen, used_sequences = [], set()
    for record in candidates:
        sequence = record["path"].stem.rsplit("_", 1)[0]
        if sequence not in used_sequences:
            chosen.append(record)
            used_sequences.add(sequence)
        if len(chosen) == limit:
            break
    return chosen

corrected_examples = choose_examples(lambda record: record["category"] == "corrected")
missed_examples = choose_examples(lambda record: record["after"]["fn"] > 0)
false_alarm_examples = choose_examples(
    lambda record: record["after"]["fp"] > 0,
    excluded={record["path"].name for record in missed_examples},
)

def draw_detection(record, moment):
    image = Image.open(record["path"]).convert("RGB")
    drawing = ImageDraw.Draw(image)
    for box in record["truth"]:
        drawing.rectangle(tuple(box), outline="lime", width=5)
    for box, confidence in zip(record[f"{moment}_boxes"], record[f"{moment}_conf"]):
        drawing.rectangle(tuple(box), outline="red", width=3)
        drawing.text((box[0], max(0, box[1]-14)), f"préd. {confidence:.2f}",
                     fill="red", stroke_width=2, stroke_fill="white")
    return image

def save_example_panel(title, examples, filename):
    if not examples:
        display(Markdown(f"**{title} :** aucun exemple dans l’échantillon de 32 images."))
        return None
    fig, axes = plt.subplots(len(examples), 2, figsize=(13, 3.8 * len(examples)), squeeze=False)
    for row, record in enumerate(examples):
        for column, moment in enumerate(("before", "after")):
            axes[row, column].imshow(draw_detection(record, moment))
            label = "AVANT" if moment == "before" else "APRÈS"
            axes[row, column].set_title(f"{label} — {describe_counts(record[moment])}")
            axes[row, column].axis("off")
    fig.suptitle(title, fontsize=16)
    fig.text(.5, .01, "VERT = annotation   •   ROUGE = prédiction", ha="center", fontsize=12)
    fig.tight_layout(rect=(0, .035, 1, .96))
    output_path = FULL_ROOT / filename
    fig.savefig(output_path, dpi=140)
    plt.close(fig)
    display(Markdown(f"### {title}"))
    display(DisplayImage(filename=str(output_path)))
    return output_path

panels = [
    ("Améliorations réussies", corrected_examples, "examples_corrected.jpg", "corrected"),
    ("Ballons encore ratés", missed_examples, "examples_missed_after.jpg", "missed_after"),
    ("Fausses alertes restantes", false_alarm_examples, "examples_false_alarms_after.jpg", "false_alarm_after"),
]
for title, examples, filename, _ in panels:
    save_example_panel(title, examples, filename)

error_report = {
    "sample_seed": 42,
    "sample_images": len(validation_paths),
    "population_images": len(all_validation_paths),
    "thresholds": {"confidence": 0.25, "iou": 0.5},
    "sample_category_counts": dict(category_counts),
    "shown_examples": [
        {"image": record["path"].name, "group": group,
         "before": record["before"], "after": record["after"]}
        for _, examples, _, group in panels for record in examples
    ],
}
(FULL_ROOT / "error_analysis.json").write_text(json.dumps(error_report, indent=2))